# Borzoi Regulatory Sequence Prediction

![Borzoi Regulatory Sequence Prediction](https://proto-bio.github.io/proto-assets/images/tool/borzoi/hero.png)

Borzoi is a long-context regulatory sequence model published by Linder et al. (2025) that predicts regulatory signals from 524,288 bp DNA input windows. It extends the Enformer architecture with larger context, higher resolution output (6,144 bins), and replicate-specific checkpoints trained on different data splits. The tool provides three functions: single-replicate prediction for fast inference with one checkpoint, ensemble prediction that runs all four replicates for robust consensus analysis with uncertainty estimates, and a gradient form of the prediction for sequence design.

This notebook demonstrates these workflows, showing how to extract selected output tracks and compute cross-replicate statistics.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("borzoi")
display_overview("borzoi")
display_docs_section("borzoi", "Background")

# Borzoi

[Borzoi](https://github.com/calico/borzoi) is a deep learning model that predicts cell-type and tissue-specific RNA-seq coverage and other regulatory genomics tracks directly from DNA sequence, developed by the Kelley lab at [Calico Life Sciences](https://www.calicolabs.com/). Given a fixed 524,288 bp input window, it returns activity across 6,144 bins of 32 bp each for thousands of human or mouse genomic assays. This toolkit exposes single-replicate prediction, a four-replicate ensemble for uncertainty estimation, and a differentiable gradient form of the same prediction.

Gene regulation acts across a wide range of genomic distances. Most promoter-proximal elements operate within a few kilobases, yet enhancers can influence genes from more than 100 kb away, and [topologically associating domains](https://en.wikipedia.org/wiki/Topologically_associating_domain) organize chromatin contacts over megabase scales. Sequence-to-function models that aim to relate noncoding variation to molecular phenotype therefore require an input window broad enough to capture these long-range relationships at fine spatial resolution.

Borzoi ([Linder et al., 2025](https://doi.org/10.1038/s41588-024-02053-6)) learns to predict cell- and tissue-specific [RNA-seq](https://en.wikipedia.org/wiki/RNA-Seq) coverage from DNA sequence, serving as a unifying model of gene regulation. Using statistics computed from its predicted coverage, Borzoi isolates and accurately scores the DNA regulatory elements that modulate transcriptional processes including transcription, splicing, and polyadenylation, with greater accuracy than comparable models. Benchmarked against state-of-the-art models, it accurately predicts the influence of variants on RNA expression and splicing and recapitulates the causal variants underlying molecular quantitative trait loci. Alongside RNA-seq, the model predicts [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression), [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq), [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq), and [histone modification](https://en.wikipedia.org/wiki/Histone_modification) tracks, making it a broad regulatory genomics predictor.

The published model couples a convolutional sequence encoder with transformer-style attention to process the full 524,288 bp window, and separate output heads produce human and mouse track predictions. The Borzoi authors trained four model replicates from independent initializations, which this toolkit exposes both as a single-replicate tool and as a four-replicate ensemble. A separate [FlashAttention](https://github.com/Dao-AILab/flash-attention)-based distillation of Borzoi, named Flashzoi, reaches comparable accuracy at substantially higher speed. The human checkpoints exposed by this toolkit use the Flashzoi distillation, and the mouse checkpoints use the standard Borzoi architecture.

## Available tools

In [2]:
display_available_tools("borzoi")

- **`run_borzoi_ensemble()`** — Regulatory activity prediction using all 4 Borzoi replicates
- **`run_borzoi_gradient()`** — Compute Borzoi track-and-bin activity gradients for relaxed DNA positions
- **`run_borzoi()`** — Regulatory activity prediction using a single Borzoi replicate

### `run_borzoi`

This function runs a single Borzoi replicate checkpoint on a 524,288 bp DNA sequence and returns a prediction matrix of shape [num_tracks, 6144]. It is the fastest option for regulatory activity prediction when cross-replicate uncertainty estimates are not required. The replicate index (0 through 3) and species (human or mouse) can be configured independently, and output tracks can be averaged into a single signal for compact downstream analysis.

In [3]:
import numpy as np
from proto_tools import (
    BORZOI_CONTEXT,
    BorzoiConfig,
    BorzoiInput,
    run_borzoi,
)

In [4]:
# Display input docs
display_api_reference("borzoi", "input", "run_borzoi")

from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

# Fetch a real human genomic window on chr11 centred on the HBB locus, exactly the
# length Borzoi requires, instead of a synthetic repeat.
_center = 5227430  # HBB midpoint, GRCh38 chr11 (NC_000011.10)
_fetch = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore", identifier="NC_000011.10", return_format="fasta",
        seq_start=_center - BORZOI_CONTEXT // 2, seq_stop=_center + BORZOI_CONTEXT // 2,
    ),
    NCBIEfetchConfig(),
)
sequence = _fetch.fasta_records[0].sequence.upper()[:BORZOI_CONTEXT]
inputs = BorzoiInput(sequences=[sequence])

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [5]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi")

single_config = BorzoiConfig(
    output_tracks=[0, 1, 2],
    species="human",
    replicate="0",
    avg_output_tracks=False,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Borzoi output tracks to extract (7611 human / 2608 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>replicate</code> | <code>Literal['0', '1', '2', '3']</code> | <code>'0'</code> | Replicate ID to run |
| <code>avg_output_tracks</code> | <code>bool</code> | <code>True</code> | Whether to average selected tracks into one output |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |

In [6]:
# Run the single-replicate prediction function
single_result = run_borzoi(inputs, single_config)

Running run_borzoi [00:00]

In [7]:
display_api_reference("borzoi", "output", "run_borzoi")

print(f"Tracks: {len(single_result.results[0].prediction)}")
print(f"Bins per track: {len(single_result.results[0].prediction[0])}")

**Output** — `BorzoiOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[BorzoiPredictionResult]</code> | required | Per-sequence Borzoi prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices used for prediction |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |
| <code>replicate</code> | <code>str</code> | required | Replicate used for prediction ('0' to '3') |
| <code>avg_output_tracks</code> | <code>bool</code> | required | Whether track outputs were averaged |

**`BorzoiPredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>prediction</code> | <code>list[list[float]]</code> | required | Predicted track activity per 32 bp bin (shape [num_tracks, 6144]) |
| <code>context_start</code> | <code>int</code> | required | 0-based start of the Borzoi input window in sequence |
| <code>context_end</code> | <code>int</code> | required | 0-based exclusive end of the Borzoi input window |
| <code>output_start</code> | <code>int</code> | required | 0-based start of the span covered by Borzoi output bins |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the Borzoi output-bin span |
| <code>output_resolution</code> | <code>int</code> | <code>32</code> | Base pairs represented by each output bin |
| <code>target_start</code> | <code>int &#124; None</code> | <code>None</code> | Requested target start, if target_ranges was provided |
| <code>target_end</code> | <code>int &#124; None</code> | <code>None</code> | Requested target end, if target_ranges was provided |

Tracks: 3
Bins per track: 6144


### `run_borzoi_ensemble`

This function runs all four Borzoi replicate checkpoints and stacks the outputs into a tensor of shape [4, num_tracks, 6144]. Access to cross-replicate mean and standard deviation provides a measure of prediction confidence: regions where replicates agree strongly (low standard deviation) are more likely to reflect genuine regulatory signal than regions with high variance. When `avg_output_tracks` is enabled, each replicate's tracks are averaged before stacking, giving a final shape of [4, 1, 6144].

In [8]:
from proto_tools import (
    BorzoiEnsembleConfig,
    run_borzoi_ensemble,
)

In [9]:
# Display input docs (same BorzoiInput as single-replicate prediction)
display_api_reference("borzoi", "input", "run_borzoi_ensemble")

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [10]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi_ensemble")

ensemble_config = BorzoiEnsembleConfig(
    output_tracks=[0, 1, 2],
    species="human",
    avg_output_tracks=True,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiEnsembleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Borzoi output tracks to extract (7611 human / 2608 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>avg_output_tracks</code> | <code>bool</code> | <code>True</code> | Whether to average selected tracks into one output |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |

In [11]:
# Run the ensemble prediction function (runs all 4 replicates)
ensemble_result = run_borzoi_ensemble(inputs, ensemble_config)

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

In [12]:
display_api_reference("borzoi", "output", "run_borzoi_ensemble")

ensemble_array = np.array(ensemble_result.results[0].predictions)
mean_pred = ensemble_array.mean(axis=0)
std_pred = ensemble_array.std(axis=0)
print(f"Replicates: {ensemble_array.shape[0]}")
print(f"Mean shape: {mean_pred.shape}")
print(f"Std max: {std_pred.max():.4f}")

**Output** — `BorzoiEnsembleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[BorzoiEnsemblePredictionResult]</code> | required | Per-sequence ensemble prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices used for prediction |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |
| <code>avg_output_tracks</code> | <code>bool</code> | required | Whether track outputs were averaged |
| <code>num_replicates</code> | <code>int</code> | <code>4</code> | Number of Borzoi replicates returned |

**`BorzoiEnsemblePredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>predictions</code> | <code>list[list[list[float]]]</code> | required | Predictions stacked across 4 model replicates (shape [4, num_tracks, 6144] at 32 bp/bin) |
| <code>context_start</code> | <code>int</code> | required | 0-based start of the Borzoi input window in sequence |
| <code>context_end</code> | <code>int</code> | required | 0-based exclusive end of the Borzoi input window |
| <code>output_start</code> | <code>int</code> | required | 0-based start of the span covered by Borzoi output bins |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the Borzoi output-bin span |
| <code>output_resolution</code> | <code>int</code> | <code>32</code> | Base pairs represented by each output bin |
| <code>target_start</code> | <code>int &#124; None</code> | <code>None</code> | Requested target start, if target_ranges was provided |
| <code>target_end</code> | <code>int &#124; None</code> | <code>None</code> | Requested target end, if target_ranges was provided |

Replicates: 4
Mean shape: (1, 6144)
Std max: 12.3535


### `run_borzoi_gradient`

The gradient tool makes Borzoi's track predictions differentiable. Each term names an output track and a span of 32 bp output bins, and comes back with its own mean activity and its own raw gradient with respect to a relaxed distribution over the bases you are designing.

Borzoi reads a fixed 524,288 bp context, but a design problem usually varies only a small part of it. A candidate therefore bundles the **whole** context sequence with the 1-indexed `positions` whose bases are relaxed and a `(len(positions), 4)` logits matrix aligned to them — so the gradient covers only the designed positions while the surrounding context still reaches the model.

In [13]:
from proto_tools import (
    BorzoiActivityTerm, BorzoiGradientCandidate, BorzoiGradientConfig,
    BorzoiGradientInput, run_borzoi_gradient,
)

display_api_reference("borzoi", "input", "run_borzoi_gradient")

# Design a 24 bp stretch at the centre of the same HBB window used above, where
# Borzoi's output bins are densest. Positions are 1-indexed and inclusive.
design_positions = [BORZOI_CONTEXT // 2 + offset for offset in range(24)]
candidate = BorzoiGradientCandidate(
    sequence=sequence,
    positions=design_positions,
    # Start from a flat distribution over A, C, G, T at each designed position.
    logits=[[0.25, 0.25, 0.25, 0.25]] * len(design_positions),
)


**Input** — `BorzoiGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[BorzoiGradientCandidate]</code> | required | One context window with relaxed positions per candidate |

In [14]:
display_api_reference("borzoi", "config", "run_borzoi_gradient")

# Differentiate track 0's activity over the output bins covering the designed window.
config = BorzoiGradientConfig(
    gradient_terms=[BorzoiActivityTerm(track=0, bin_start=3060, bin_end=3085)],
)

gradient = run_borzoi_gradient(BorzoiGradientInput(inputs=candidate), config)


**Config** — `BorzoiGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[BorzoiActivityTerm]</code> | <code>[BorzoiActivityTerm(id='track_activity', track=0, bin_start=1, bin_end=6144)]</code> | Track-and-bin quantities to differentiate, by name or as instances |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>replicate</code> | <code>Literal['0', '1', '2', '3']</code> | <code>'0'</code> | Replicate ID to run |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

Running run_borzoi_gradient [00:00]

In [15]:
display_api_reference("borzoi", "output", "run_borzoi_gradient")

# One result per candidate; this call passed one.
result = gradient.results[0]
print(f"Designed positions: {result.metrics['designed_positions']}")
for term in result.terms:
    print(f"  track {term.term.track} bins {term.term.bin_start}-{term.term.bin_end}: "
          f"mean activity {term.term_value:.4f}, "
          f"gradient shape ({len(term.gradient)}, {len(term.gradient[0])})")
display_api_reference("borzoi", "input", "run_borzoi_gradient")
display_api_reference("borzoi", "config", "run_borzoi_gradient")
display_api_reference("borzoi", "output", "run_borzoi_gradient")


**Output** — `BorzoiGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[BorzoiGradientResult]</code> | required | One result per input candidate, in input order, each carrying its track activities. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`BorzoiGradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>terms</code> | <code>list[BorzoiTermGradient]</code> | <code>[]</code> | One gradient per requested track-and-bin quantity, in the order requested. |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**`BorzoiTermGradient`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>term</code> | <code>BorzoiActivityTerm</code> | required | Track-and-bin quantity answered. |
| <code>term_value</code> | <code>float</code> | required | Raw scalar the model produced for this quantity. |
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Raw derivative of term_value w.r.t. the input; matches its shape and column order. |
| <code>grad_l2_norm</code> | <code>float &#124; None</code> | <code>None</code> | Euclidean norm of the flattened gradient; computed automatically when a gradient is present. |

**`BorzoiActivityTerm`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>Literal['track_activity']</code> | <code>'track_activity'</code> | Identifies this quantity as Borzoi mean track activity over a bin span. |
| <code>track</code> | <code>int</code> | <code>0</code> | Borzoi output-track index to optimize (7611 human / 2608 mouse tracks) |
| <code>bin_start</code> | <code>int</code> | <code>1</code> | First output bin to average, 1-indexed and inclusive |
| <code>bin_end</code> | <code>int</code> | <code>6144</code> | Last output bin to average, 1-indexed and inclusive |

**`GradientMetadata`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>description</code> | <code>str</code> | <code>''</code> | What this scalar quantity measures. |
| <code>units</code> | <code>str &#124; None</code> | <code>None</code> | Physical units of the value, where it has them. |
| <code>suggested_optimization_direction</code> | <code>Literal['higher', 'lower', 'in-range', 'context-dependent']</code> | <code>'context-dependent'</code> | Advisory direction for this quantity; does not affect the sign of the gradient. |
| <code>vocab</code> | <code>list[str]</code> | required | Column identity and order for this gradient. |
| <code>coordinate_space</code> | <code>Literal['logits', 'probabilities']</code> | required | Whether the derivative is w.r.t. logits or probabilities. |
| <code>length</code> | <code>int</code> | required | Number of differentiated rows. |
| <code>positions</code> | <code>list[int] &#124; None</code> | <code>None</code> | 1-indexed input position per row; None means contiguous 1..length. |
| <code>input_fingerprint</code> | <code>str</code> | required | Digest of the differentiated matrix, identifying the point the gradient was taken at. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient in force. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Relaxation blend in force. |

**Metrics** — `BorzoiGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>designed_positions</code> **(primary)** | <code>int</code> | <code>&gt;= 1</code> |  | Number of relaxed context positions this candidate carries. |

Designed positions: 24
  track 0 bins 3060-3085: mean activity 0.0213, gradient shape (24, 4)


**Input** — `BorzoiGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[BorzoiGradientCandidate]</code> | required | One context window with relaxed positions per candidate |

**Config** — `BorzoiGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[BorzoiActivityTerm]</code> | <code>[BorzoiActivityTerm(id='track_activity', track=0, bin_start=1, bin_end=6144)]</code> | Track-and-bin quantities to differentiate, by name or as instances |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>replicate</code> | <code>Literal['0', '1', '2', '3']</code> | <code>'0'</code> | Replicate ID to run |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

**Output** — `BorzoiGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[BorzoiGradientResult]</code> | required | One result per input candidate, in input order, each carrying its track activities. |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`BorzoiGradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>terms</code> | <code>list[BorzoiTermGradient]</code> | <code>[]</code> | One gradient per requested track-and-bin quantity, in the order requested. |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**`BorzoiTermGradient`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>term</code> | <code>BorzoiActivityTerm</code> | required | Track-and-bin quantity answered. |
| <code>term_value</code> | <code>float</code> | required | Raw scalar the model produced for this quantity. |
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Raw derivative of term_value w.r.t. the input; matches its shape and column order. |
| <code>grad_l2_norm</code> | <code>float &#124; None</code> | <code>None</code> | Euclidean norm of the flattened gradient; computed automatically when a gradient is present. |

**`BorzoiActivityTerm`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>id</code> | <code>Literal['track_activity']</code> | <code>'track_activity'</code> | Identifies this quantity as Borzoi mean track activity over a bin span. |
| <code>track</code> | <code>int</code> | <code>0</code> | Borzoi output-track index to optimize (7611 human / 2608 mouse tracks) |
| <code>bin_start</code> | <code>int</code> | <code>1</code> | First output bin to average, 1-indexed and inclusive |
| <code>bin_end</code> | <code>int</code> | <code>6144</code> | Last output bin to average, 1-indexed and inclusive |

**`GradientMetadata`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>description</code> | <code>str</code> | <code>''</code> | What this scalar quantity measures. |
| <code>units</code> | <code>str &#124; None</code> | <code>None</code> | Physical units of the value, where it has them. |
| <code>suggested_optimization_direction</code> | <code>Literal['higher', 'lower', 'in-range', 'context-dependent']</code> | <code>'context-dependent'</code> | Advisory direction for this quantity; does not affect the sign of the gradient. |
| <code>vocab</code> | <code>list[str]</code> | required | Column identity and order for this gradient. |
| <code>coordinate_space</code> | <code>Literal['logits', 'probabilities']</code> | required | Whether the derivative is w.r.t. logits or probabilities. |
| <code>length</code> | <code>int</code> | required | Number of differentiated rows. |
| <code>positions</code> | <code>list[int] &#124; None</code> | <code>None</code> | 1-indexed input position per row; None means contiguous 1..length. |
| <code>input_fingerprint</code> | <code>str</code> | required | Digest of the differentiated matrix, identifying the point the gradient was taken at. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient in force. |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Relaxation blend in force. |

**Metrics** — `BorzoiGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>designed_positions</code> **(primary)** | <code>int</code> | <code>&gt;= 1</code> |  | Number of relaxed context positions this candidate carries. |

## Export Results

Single-replicate and ensemble outputs can be saved to standard file formats for downstream analysis. Supported formats include JSON and CSV.

In [16]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

single_result.export(name="borzoi_single", export_path=output_dir, file_format="json")
print(f"Single-replicate result exported to {output_dir}/borzoi_single.json")

ensemble_result.export(name="borzoi_ensemble", export_path=output_dir, file_format="csv")
print(f"Ensemble summary exported to {output_dir}/borzoi_ensemble.csv")

Single-replicate result exported to example_output/borzoi_single.json
Ensemble summary exported to example_output/borzoi_ensemble.csv
